# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Load starter data
candidates = [
    Path.cwd() / "data" / "raw" / "content_refresh_anonymized.csv",
    Path.cwd().parent / "data" / "raw" / "content_refresh_anonymized.csv",
    Path.cwd().parent.parent / "data" / "raw" / "content_refresh_anonymized.csv",
]
data_path = next((c for c in candidates if c.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")

starter = pd.read_csv(data_path)
print(f"Loaded: {data_path.name}")
print(f"Rows: {len(starter):,}")
print(f"Unique content items: {starter['content_id'].nunique():,}")
print(f"Unique clients: {starter['client_id'].nunique():,}")

In [ ]:
# Build feature vector following the pipeline logic from scripts/01_prepare_features.py
# and ml_utils.py MODEL_NUMERIC_FEATURES / MODEL_CATEGORICAL_FEATURES

df = starter.copy()

# --- Target (for reference only — NEVER a feature) ---
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# --- Engineered numeric features (log1p of heavy-tailed counts) ---
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

# --- Binary flags for missingness / presence ---
df["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
df["measurable_opportunity"] = ((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)).astype(int)

# --- Numeric fills: fill blanks with 0 (numerics) ---
numeric_features = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"
]
for col in numeric_features:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# --- Categorical fills: fill blanks with 'unknown' ---
categorical_features = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "char_count_tier", "impression_tier", "position_tier"
]
for col in categorical_features:
    if col in df.columns:
        df[col] = df[col].fillna("unknown")

# --- Final feature matrix (drop IDs, label sources, non-features) ---
feature_cols_numeric = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

feature_cols_categorical = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier"
]

feature_cols = feature_cols_numeric + feature_cols_categorical
X = df[feature_cols].copy()
y = df["is_declining_label"].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Numeric features: {len(feature_cols_numeric)}")
print(f"Categorical features: {len(feature_cols_categorical)}")
print(f"Target distribution:\n{y.value_counts().rename('count').to_frame()}")
print(f"Decline rate: {y.mean():.1%}")

# Show feature dtypes
feature_dtypes = X.dtypes.rename("dtype").to_frame()
feature_dtypes["missing"] = X.isna().sum()
feature_dtypes

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# Feature notes table
feature_notes = pd.DataFrame({
    "feature": feature_cols,
    "type": ["numeric"] * len(feature_cols_numeric) + ["categorical"] * len(feature_cols_categorical),
    "meaning": [
        "Keyword search volume estimate (0 if no keyword data)",
        "Keyword competition score 0-1 (0 if no keyword data)",
        "Keyword cost-per-click estimate (0 if no keyword data)",
        "Article word count (0 for 7,699 rows not measured)",
        "Article character count (0 alongside word_count)",
        "log1p(impressions_90d) — search demand signal",
        "log1p(clicks_90d) — search click signal",
        "log1p(sessions_90d) — engagement signal",
        "log1p(ai_sessions_90d) — AI referral signal (sparse)",
        "Days with ≥1 impression in 90d window (0-90)",
        "Days with ≥1 session in 90d window (0-90)",
        "Days since content creation (≥90 in this slice)",
        "Days since last content update",
        "Click-through rate ×100 (0.76 = 0.76%)",
        "Average GSC position (0 = no data, not rank 0)",
        "Engaged sessions / sessions ×100",
        "Scroll events / pageviews ×100 (can exceed 100)",
        "AI sessions / sessions ×100 (can exceed 100)",
        "Keyword competition bucket: LOW/MEDIUM/HIGH/unknown",
        "Content type: keyword article / feedly article / comparison article",
        "Search intent: informational / transactional / commercial / navigational / unknown",
        "Age bucket from content_age_days",
        "Freshness bucket from days_since_last_update",
        "Word count bucket (<1000/1000-2000/2000-3500/3500+/unknown)",
        "Impression tier (no_data/none/low/moderate/good/excellent)",
        "Position tier (no_data/top_3/page_1/striking/page_3_5/deep)",
    ],
    "missing_handling": [
        "fillna(0) — missing = no keyword data",
        "fillna(0) — missing = no keyword data",
        "fillna(0) — missing = no keyword data",
        "fillna(0) — not measured for 25.7% rows",
        "fillna(0) — not measured for 25.7% rows",
        "computed from impressions_90d (no missing)",
        "computed from clicks_90d (no missing)",
        "computed from sessions_90d (no missing)",
        "computed from ai_sessions_90d (no missing)",
        "no missing (0-90 range)",
        "no missing (0-90 range)",
        "no missing (all ≥90)",
        "no missing",
        "fillna(0) — rate, 0 when impressions=0",
        "fillna(0) — 0 means 'no position data' (1,205 rows)",
        "fillna(0) — 0 when sessions=0",
        "fillna(0) — 0 when pageviews=0",
        "fillna(0) — 0 when sessions=0",
        "fillna('unknown') — missing = no keyword data",
        "no missing",
        "fillna('unknown') — missing when intent unknown",
        "no missing (derived from age)",
        "no missing (derived from freshness)",
        "fillna('unknown') — missing when word_count missing",
        "no missing (derived from impressions)",
        "no missing (derived from avg_position)",
    ],
    "available_before_prediction": ["yes"] * len(feature_cols),
})

feature_notes

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# --- LEAKAGE TEST 1: Label-derived features ---
# The label is is_declining_label = (trend_direction == 'down')
# trend_direction and trend_pct are the SOURCE of the label — must NOT be features

print("=== LEAKAGE TEST 1: Label-derived columns ===")
print(f"trend_direction in feature_cols: {'trend_direction' in feature_cols}")
print(f"trend_pct in feature_cols: {'trend_pct' in feature_cols}")
print(f"is_declining_label in feature_cols: {'is_declining_label' in feature_cols}")

# Demonstrate what happens if we DO include trend_direction (the leaky version)
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Clean feature matrix (no leakage)
X_clean = X.copy()

# Leaky feature matrix (includes trend_direction)
X_leaky = df[feature_cols + ["trend_direction"]].copy()

# Quick model test
def quick_score(X_mat, y_vec, name):
    cat_cols = [c for c in feature_cols_categorical if c in X_mat.columns]
    num_cols = [c for c in feature_cols_numeric if c in X_mat.columns]
    if "trend_direction" in X_mat.columns:
        cat_cols = cat_cols + ["trend_direction"]

    preprocessor = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), cat_cols),
        ("num", "passthrough", num_cols),
    ])
    clf = Pipeline([
        ("prep", preprocessor),
        ("clf", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)),
    ])
    # Use client-grouped CV would be better, but this is a quick demo
    scores = cross_val_score(clf, X_mat, y_vec, cv=3, scoring="roc_auc")
    print(f"{name}: ROC-AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")
    return scores.mean()

auc_clean = quick_score(X_clean, y, "Clean features (no trend_direction)")
auc_leaky = quick_score(X_leaky, y, "Leaky features (WITH trend_direction)")
print(f"\nAUC jump from leakage: {auc_leaky - auc_clean:.4f}")
print(">>> This confirms trend_direction is label-derived and MUST be excluded.")

In [ ]:
# --- LEAKAGE TEST 2: Future/overlapping windows ---
# The data is a 90-day trailing window. The label uses:
#   trend_pct = (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d * 100
# where impressions_last_30d is the MOST RECENT 30 days.
# Features that include impressions_last_30d / clicks_last_30d / sessions_last_30d
# would leak the label window into features.

print("=== LEAKAGE TEST 2: Future/overlapping windows ===")
print("Label window: impressions_last_30d vs impressions_prev_30d (days 31-60 back)")
print()

# Check which 30-day columns exist in our feature set
window_cols = ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
               "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
for col in window_cols:
    in_features = col in feature_cols
    print(f"  {col}: in feature_cols = {in_features}")

print()
print("SAFE: impressions_prev_30d, clicks_prev_30d, sessions_prev_30d (days 31-60 back)")
print("UNSAFE: impressions_last_30d, clicks_last_30d, sessions_last_30d (days 1-30 back — overlaps label)")
print("Our feature set uses 90-day aggregates and PREV_30d windows only — no overlap.")

In [ ]:
# --- LEAKAGE TEST 3: Product flags / decision-derived features ---
# The data dictionary explicitly states:
# - provider_used, model_used are NOT model features
# - FlyRank's product scores (health_score, priority_score, action_type, refresh_tier)
#   are NOT in this dataset at all (observable signals only)

print("=== LEAKAGE TEST 3: Product flags / decision-derived features ===")

product_like_cols = ["provider_used", "model_used", "health_score", "priority_score",
                     "action_type", "refresh_tier", "refresh_flag"]
for col in product_like_cols:
    in_data = col in df.columns
    in_features = col in feature_cols
    print(f"  {col}: in raw data = {in_data}, in feature_cols = {in_features}")

print()
print("provider_used and model_used exist in raw data but are EXCLUDED from features.")
print("Product scores (health_score, priority_score, action_type, refresh_tier) are NOT in this dataset.")
print("This dataset ships OBSERVABLE SIGNALS ONLY — by design to prevent circular results.")

In [ ]:
# --- LEAKAGE TEST 4: Grouped split sanity check ---
# Client-holdout validation: pages from same client share patterns.
# Random split inflates scores; grouped split is honest.

from sklearn.model_selection import GroupKFold

print("=== LEAKAGE TEST 4: Grouped vs Random split ===")

cat_cols = feature_cols_categorical
num_cols = feature_cols_numeric

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), cat_cols),
    ("num", "passthrough", num_cols),
])
clf = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)),
])

# Random 3-fold CV
from sklearn.model_selection import cross_val_score
random_scores = cross_val_score(clf, X_clean, y, cv=3, scoring="roc_auc")
print(f"Random 3-fold CV ROC-AUC: {random_scores.mean():.4f} (+/- {random_scores.std():.4f})")

# Client-grouped 3-fold CV
groups = df["client_id"]
gkf = GroupKFold(n_splits=3)
grouped_scores = cross_val_score(clf, X_clean, y, groups=groups, cv=gkf, scoring="roc_auc")
print(f"Client-grouped 3-fold CV ROC-AUC: {grouped_scores.mean():.4f} (+/- {grouped_scores.std():.4f})")
print(f"Gap (random - grouped): {random_scores.mean() - grouped_scores.mean():.4f}")
print()
print("A gap > 0.02 suggests the model was memorizing client patterns.")
print("Client-grouped is the HONEST estimate for deployment on new clients.")

In [ ]:
# --- LEAKAGE TEST 5: Base rate and top feature sanity ---
print("=== LEAKAGE TEST 5: Base rate ===")
print(f"Decline rate (base rate): {y.mean():.1%}")
print(f"Majority class accuracy: {max(y.mean(), 1-y.mean()):.1%}")
print()
print("Any model ROC-AUC near 1.0 with this base rate is suspicious.")
print("Random Forest on clean features got ~0.75 AUC — reasonable, not perfect.")
print()

# Top feature importance sanity check using the pipeline directly
from sklearn.inspection import permutation_importance

# Fit pipeline on full data for importance inspection
clf_full = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)),
])
clf_full.fit(X_clean, y)

# Permutation importance on a sample (using original feature matrix)
sample_idx = np.random.choice(len(X_clean), size=min(5000, len(X_clean)), replace=False)
result = permutation_importance(clf_full, X_clean.iloc[sample_idx], y.iloc[sample_idx],
                                n_repeats=5, random_state=42, n_jobs=-1)

# permutation_importance returns one importance per ORIGINAL feature column (26)
# Use feature_cols as the names (not one-hot expanded names)
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": result.importances_mean,
    "importance_std": result.importances_std,
}).sort_values("importance_mean", ascending=False).head(15)

print("Top 15 features by permutation importance:")
print(importance_df.to_string(index=False))

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# Excluded fields with reasons
excluded = pd.DataFrame({
    "column": [
        "content_id",
        "client_id",
        "trend_direction",
        "trend_pct",
        "is_declining_label",
        "provider_used",
        "model_used",
        "impressions_last_30d",
        "clicks_last_30d",
        "sessions_last_30d",
        "pageviews_90d",
        "users_90d",
        "engaged_sessions_90d",
        "scroll_events_90d",
        "ai_sessions_90d",
        "clicks_90d",
        "impressions_90d",
        "sessions_90d",
        "age_tier_order",
        "char_count_tier",
        "competition_level",  # note: this IS used, but raw competition is numeric
    ],
    "reason": [
        "Pseudonym ID — grouping/joins only, no signal",
        "Pseudonym ID — grouping/joins only, used for GroupKFold splits",
        "LABEL SOURCE — defines is_declining_label, using it is direct leakage",
        "LABEL SOURCE — trend_pct computes trend_direction, using it is direct leakage",
        "TARGET — this is what we predict, never a feature",
        "LLM provider metadata — not an observable search/engagement signal",
        "LLM model metadata — not an observable search/engagement signal",
        "FUTURE WINDOW — overlaps label window (days 1-30), leakage risk",
        "FUTURE WINDOW — overlaps label window (days 1-30), leakage risk",
        "FUTURE WINDOW — overlaps label window (days 1-30), leakage risk",
        "Redundant with sessions_90d / impressions_90d aggregates; not in model features",
        "Redundant with sessions_90d; not in model features",
        "Redundant with sessions_90d; not in model features",
        "Redundant with days_with_sessions / scroll_rate; not in model features",
        "Redundant with ai_traffic_pct + has_ai_sessions; not in model features",
        "Raw count — heavy-tailed, use log_clicks_90d instead",
        "Raw count — heavy-tailed, use log_impressions_90d instead",
        "Raw count — heavy-tailed, use log_sessions_90d instead",
        "Ordinal encoding of age_tier — use age_tier categorical instead",
        "Redundant with word_count_tier; word_count_tier is more interpretable",
        "USED as categorical (competition_level), raw competition numeric is 0 for missing keyword data",
    ],
})

excluded

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.